# Ablation-Based Modality Importance Demo

**Amaç:** Mock data üzerinde `compute_ablation_scores` çalıştırıp her hastalık grubu için
hangi modalite (EEG, ECG, Resp, EMG) en önemli olduğunu heatmap ile görselleştirmek.

**Yöntem:** Her modaliteyi sıfırlayıp KNN similarity drop ölçüyoruz → büyük drop = o modalite önemli.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from mock_data import generate_mock_embeddings
from config import DISEASE_NAMES, MODALITY_DIM_RANGES
from similarity_engine import build_reference_index
from pipeline import compute_ablation_scores

print('Imports OK')

## 1. Mock Data Oluştur

In [ ]:
np.random.seed(42)
embeddings, labels = generate_mock_embeddings(n_samples=500, seed=42)
print(f'Embeddings: {embeddings.shape}, Labels: {np.unique(labels)}')

## 2. KNN İndeksi Kur & Her Hastalık İçin Ablation Skoru Hesapla

In [ ]:
index = build_reference_index(embeddings, n_neighbors=10)

# Her hastalık grubundan 1 representative hasta seç (self-exclusion ile)
results = {}  # disease_name -> {modality: score}

for disease_id, disease_name in sorted(DISEASE_NAMES.items()):
    # O hastalık grubundan ilk hastayı query olarak al
    group_mask = labels == disease_id
    if not group_mask.any():
        continue
    query_idx = np.where(group_mask)[0][0]
    query_emb = embeddings[query_idx]
    
    # Self-exclusion: query haricindeki hastalarla index kur
    ref_mask = np.ones(len(embeddings), dtype=bool)
    ref_mask[query_idx] = False
    ref_index = build_reference_index(embeddings[ref_mask], n_neighbors=10)
    
    scores = compute_ablation_scores(
        query_embedding=query_emb,
        reference_index=ref_index,
        reference_labels=labels[ref_mask],
    )
    results[disease_name] = scores
    print(f'{disease_id:2d}. {disease_name:40s} | EEG={scores["eeg"]:.4f}  ECG={scores["ecg"]:.4f}  Resp={scores["resp"]:.4f}  EMG={scores["emg"]:.4f}')

print(f'\n{len(results)} hastalık grubu analiz edildi.')

## 3. Heatmap: Hastalık × Modalite Ablation Skorları

In [ ]:
# DataFrame oluştur
df = pd.DataFrame(results).T  # rows=diseases, cols=modalities
df.columns = [m.upper() for m in df.columns]
df = df.sort_index()

# Heatmap
fig, ax = plt.subplots(figsize=(8, 10))
sns.heatmap(
    df,
    annot=True,
    fmt='.4f',
    cmap='YlOrRd',
    linewidths=0.5,
    cbar_kws={'label': 'Similarity Drop (ablation score)'},
    ax=ax,
)
ax.set_title('Modality Importance per Disease\n(Mask-Based Ablation — Similarity Drop)', 
             fontsize=13, fontweight='bold', pad=15)
ax.set_ylabel('Disease Group', fontsize=11)
ax.set_xlabel('Modality', fontsize=11)
plt.tight_layout()

# Kaydet
os.makedirs('../outputs/figures', exist_ok=True)
fig.savefig('../outputs/figures/ablation_heatmap.png', dpi=150, bbox_inches='tight')
print('Heatmap saved to outputs/figures/ablation_heatmap.png')
plt.show()

## 4. Her Hastalık İçin En Dominant Modalite

In [ ]:
# En yüksek ablation skoru olan modalite = en önemli
dominant = df.idxmax(axis=1).to_frame('Dominant Modality')
dominant['Score'] = df.max(axis=1)
dominant['2nd Modality'] = df.apply(lambda row: row.nlargest(2).index[1], axis=1)
dominant['2nd Score'] = df.apply(lambda row: row.nlargest(2).values[1], axis=1)
dominant['Gap'] = dominant['Score'] - dominant['2nd Score']

print(dominant.to_string())
print(f'\nOrtalama dominant-vs-2nd gap: {dominant["Gap"].mean():.4f}')

## 5. Bar Chart: Modalite Başına Ortalama Önem

In [ ]:
mean_scores = df.mean(axis=0).sort_values(ascending=False)

fig2, ax2 = plt.subplots(figsize=(6, 4))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
mean_scores.plot(kind='bar', ax=ax2, color=colors[:len(mean_scores)], edgecolor='black')
ax2.set_title('Average Modality Importance (across all diseases)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Mean Ablation Score (similarity drop)')
ax2.set_xlabel('')
ax2.tick_params(axis='x', rotation=0)

for i, (idx, val) in enumerate(mean_scores.items()):
    ax2.text(i, val + 0.001, f'{val:.4f}', ha='center', fontsize=9)

plt.tight_layout()
fig2.savefig('../outputs/figures/ablation_bar_chart.png', dpi=150, bbox_inches='tight')
print('Bar chart saved to outputs/figures/ablation_bar_chart.png')
plt.show()